<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Aprendizaje Automático Avanzado</h1>
    <h3>Repaso: Aprendizaje No Supervisado</h3>
    
</div>


> Repaso del **aprendizaje no supervisado**: descubrir estructura oculta en los datos **sin etiquetas**, mediante **reducción de dimensionalidad (PCA)** y **clustering**.


### Contenido

- Introducción
- Principal Components Analysis (PCA)
- Clustering
    - K-Means
    - Mixture Models

<sub>Material adaptado de las clases prácticas de Inteligencia Artificial — UTN, Facultad Regional Villa María.</sub>


### Panorama del aprendizaje no supervisado

```mermaid
flowchart TD
    U([Aprendizaje No Supervisado])
    U --> R[Reducción de dimensión]
    U --> C[Clustering]
    R --> PCA[PCA]
    C --> KM[K-Means]
    C --> MM[Mixture Models - GMM]

    classDef root fill:#000047,color:#2E9AFE,stroke:#000047;
    class U root;
```


## Introducción

* Los métodos de aprendizaje no supervisado apuntan realizar un análisis _exploratorio_ de los datos, para poder descubrir aspectos de interés sobre los mismos dejando de lado su posible asociación con una respuesta $y$.
* El conjunto de datos que se analiza es la matriz $X$, con el fin de extraer modelos o conocimiento sobre los datos de la misma. Debido a que no incluimos un conjunto $y$ de salidas para aprender iterativamente, para esta tarea se utilizan técnicas distintas.
* No obstante, para ciertos problemas es posible combinar el conocimiento extraído a partir de este tipo de aprendizaje para mejorar los modelos de aprendizaje supervisado vistos anteriormente.
* Algunos ejemplos de tarea de aprendizaje no supervisado:
    * Dada una colección de imágenes, agrupar aquellas similares.
    * Dadas varias fuentes de sonido en una misma pista de audio, como por ejemplo una persona hablando por teléfono y un parlante emitiendo música, obtener la separación de las distintas fuentes.
    * Entender cómo se correlacionan los features de un determinado conjunto de datos.
* En la presente clase vamos a ver la implementación de dos técnicas muy utilizadas: clústering y PCA.



## Principal Components Analysis (PCA)

* En la práctica, los features $X$ de un dataset suelen tener un grado de relación entre sí (ej. pensar en features edad y altura para una muestra de datos de niños y adolescentes).

* Como muchas veces, para estos datasets con datos correlacionados, es baja la cantidad de features que definen al dataset, debería ser posible encontrar la forma de explicar la mayor parte posible del dataset con la menor cantidad de features posible. 

* El *análisis de componentes principales (PCA)* realiza esto al calcular features transformados que surgen como combinación lineal de los demás, reduciendo así la dimensión del dataset y por ende el efecto de la *Curse of Dimensionality*.

* En otras palabras, PCA apunta a reducir la dimensionalidad de un dataset al considerar qué features explican mejor la varianza de los datos.

* Para ello se crea una proyección linear del dataset. Los datos correlacionados se obtienen mediante un número menor de variables representativas no correlacionadas (ortogonales) que expliquen de forma conjunta la varianza del set original.


### Repaso rápido:

* Formalmente, PCA busca obtener las direcciones principales que maximizan la varianza en el conjunto de datos.

* Dado un conjunto de datos $X$ de orden $n \times p$ ($n$ observaciones y $p$ predictores), se define como _primer componente principal_ a la combinación lineal normalizada para cada fila de observaciones $$z_{i1} = \phi_{11}X_{i1} + \phi_{21}X_{i2} + ... + \phi_{p1}X_{ip}, \forall i=1, 2, ..., n$$ que captura la mayor varianza para la suma de las observaciones. 

* Para normalizar, $\forall i, \sum_{j=1}^p \phi_{ji}^2 = 1$ donde también $\forall j, \frac{1}{n}\sum_{i=1}^n x_{ij} = 0$ (por ejemplo mediante z-score)

* Es decir que el primer componente principal está dado por el vector $\phi_{11}, \phi_{21}, ..., \phi_{p1}$ que cumple con
$$\arg\max_{\phi_{11}, \phi_{12}, ..., \phi_{1p}} \Bigg(\frac{1}{n} \sum_{i=1}^n \Big(\sum_{j=1}^p \phi_{j1}X_{ij} \Big)^2 \Bigg)$$

(notar que en la fórmula de varianza no se está restando el valor de la media puesto que, al estar normalizada, la misma es 0)

* El _segundo componente principal_ se define de forma similar, con la restricción adicional de que el mismo debe ser ortogonal al primer componente, es decir que el producto punto de ambos vectores es 0.

* Esta restricción hace que el primer PC sea aquel que mayor varianza abarque, seguido por el segundo PC que capturará la segunda mayor varianza, y así sucesivamente. Además, limita a una cantidad máxima de $p$ de PC que pueden ser obtenidos.

* El vector formado por $\phi_1 = (\phi_{11}, \phi_{21}, ..., \phi_{p1})^T$ se denomina _first principal component loading vector_; mientras que $z_{11}, z_{21}, ..., z_{n1}$ son los _scores_ del primer componente principal.

Por ejemplo, para el caso de dos dimensiones $x_1$ y $x_2$

![](Figures/pca_projection.gif)

* El primer PC está dado por la combinación lineal $\phi_{11} x_1 + \phi_{12} x_2$, que forma el segmento más largo, mientras que el segundo PC está dado por $\phi_{21} x_1 + \phi_{22} x_2$.

* A mayor varianza maximizada aplicando PCA, mayor dispersión de los puntos rojos resultantes de la proyección ortogonal de cada uno de los datos.

* Como puede apreciarse, maximizar la varianza **equivale a minimizar el error cuadrático dado por la distancia ortogonal entre proyección lineal y los puntos de datos**.


* Veamos un ejemplo de su aplicación para el **dataset Iris...**

In [ ]:
# Recordemos el Iris dataset...

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.datasets import load_iris
from sklearn import neighbors

iris = load_iris()

# tomamos los primeros dos features para mostrarlos mejor graficamente (el largo y el ancho del sepalo)
X = iris.data[:, :2]
y = iris.target

plot = plt.scatter(X[y==0, 0], X[y==0, 1], label=iris.target_names[0], color='orange')
plot = plt.scatter(X[y==1, 0], X[y==1, 1], label=iris.target_names[1], color='blue')
plot = plt.scatter(X[y==2, 0], X[y==2, 1], label=iris.target_names[2], color='green')

plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.legend(loc='best', numpoints=1)
plt.show()

In [ ]:
import numpy as np

from sklearn.preprocessing import scale
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

iris = load_iris()
X = iris.data
y = iris.target

#Insanciando un objeto de PCA


# Antes de transformar los datos, los mismos deben estar normalizados
# scale() estandariza los datos con respecto a la media 0 y a la desv. estándar 1 (z-score)


# X_transformed son los datos X transformados linealmente con respecto a los componentes principales


# Veamos los vectores de componentes de PCA...

print('Componentes de PCA (ordenados desc. desde el 1° hasta el 4to vector): \n\n', pca.components_) 
# Notar que, por la restricción de la ortogonalidad, la máxima
# cantidad de componentes principales es la de los features de X


print('Varianza explicada por cada componente: \n\n', pca.explained_variance_)

In [ ]:
# Visualizamos ahora cuánto es explicada la varianza
# por cada uno de los componentes principales


# pca.explained_variance_ratio_ es quien nos devuelve el gráfico de la varianza

plt.bar(y_pos, np.round(100 * pca.explained_variance_ratio_,
                        decimals=1), align='center', alpha=0.5)
plt.xticks(y_pos, np.arange(1, n_components+1))
plt.xlabel('N° de Componente Principal')
plt.ylabel('Varianza')
plt.title('Varianza explicada por cada componente')
plt.show()

* ¿Qué significa esto? Significa que el primer componente principal que generamos explica más de un 70% de la varianza de los datos. En otras palabras, **al usar una varible (transformada) para graficar tenemos el 70% de la información acerca de los features que representan las 4 variables originales**.

* Notar que un componente principal es una combinación lineal de los valores de los distintos features; **no confundir el número del componente principal con el número de feature, cada feature transformado "resume info" de varios features**.

* Veamos esto gráficamente...

In [ ]:
plt.plot(X_transformed[y==0,0], np.zeros(len(X_transformed[y==0,0])), 'o', label=iris.target_names[0], color='orange')
plt.plot(X_transformed[y==1,0], np.zeros(len(X_transformed[y==1,0])), 'o', label=iris.target_names[1], color='blue')
plt.plot(X_transformed[y==2,0], np.zeros(len(X_transformed[y==2,0])), 'o', label=iris.target_names[2], color='green')
plt.xlabel('Valor del Primer Componente Principal')
plt.legend(loc='best', numpoints=1)
plt.show()


Incluimos el segundo componente principal, vemos que no cambia sustancialmente...

In [ ]:
plot = plt.scatter(X_transformed[y==0, 0], X_transformed[y==0, 1], label=iris.target_names[0], color='orange')
plot = plt.scatter(X_transformed[y==1, 0], X_transformed[y==1, 1], label=iris.target_names[1], color='blue')
plot = plt.scatter(X_transformed[y==2, 0], X_transformed[y==2, 1], label=iris.target_names[2], color='green')
plt.xlabel('Valor del Primer Componente Principal')
plt.ylabel('Valor del Segundo Componente Principal')
plt.legend(loc='best', numpoints=1)
plt.show()

Agregamos los loading vectors de cada feature para ver cómo los mismos varian segun el valor de los 2 primeros componentes...

(parte del codigo basado en https://stackoverflow.com/questions/42281966/how-to-plot-vectors-in-python-using-matplotlib)

In [ ]:
# codigo agregado para graficar los vectores
V = np.array([[pca.components_[0,0],pca.components_[1,0]],
              [pca.components_[0,1],pca.components_[1,1]],
              [pca.components_[0,2],pca.components_[1,2]],
              [pca.components_[0,3],pca.components_[1,3]]])
origin = [0,0,0,0],[0,0,0,0] # origen, desde donde se situaran los vectores

plot = plt.scatter(X_transformed[y==0, 0], X_transformed[y==0, 1], label=iris.target_names[0], color='orange')
plot = plt.scatter(X_transformed[y==1, 0], X_transformed[y==1, 1], label=iris.target_names[1], color='blue')
plot = plt.scatter(X_transformed[y==2, 0], X_transformed[y==2, 1], label=iris.target_names[2], color='green')

# notar que los vectores están agrandados a modo de mejor visualización
plt.quiver(*origin, V[:,0], V[:,1], color=['r','purple','grey','cyan'], scale=3)

# se agrega texto a los vectores, quitando el substring " (cm)"
plt.text(pca.components_[0,0],pca.components_[1,0], iris.feature_names[0].replace(' (cm)', ''), fontsize=12, weight=1000)
plt.text(pca.components_[0,1],pca.components_[1,1], iris.feature_names[1].replace(' (cm)', ''), fontsize=12, weight=1000)
plt.text(pca.components_[0,2],pca.components_[1,2], iris.feature_names[2].replace(' (cm)', ''), fontsize=12, weight=1000)
plt.text(pca.components_[0,3],pca.components_[1,3], iris.feature_names[3].replace(' (cm)', ''), fontsize=12, weight=1000)
plt.xlabel('Valor del Primer Componente Principal')
plt.ylabel('Valor del Segundo Componente Principal')
plt.legend(loc='best', numpoints=1)
plt.show()

Por ejemplo, podemos ver intuitivamente que el vector del feature "ancho del sépalo" (**"sepal width"**) crece en mayor proporción a medida que crece el segundo componente principal, mientras que también crece (pero en menor medida) de forma inversamente proporcional al valor del primer componente principal (es decir, cuando el mismo decrece).

**Algunas utilidades de PCA**

1. Buena forma de visualización de los datos, al poder apreciar cómo los features están correlacionados con las distintas observaciones.

2. Reducción de la dimensionalidad: por ejemplo, si quisiéramos hacer un clasificador en un dataset muy grande y vemos que sólo unos pocos features transformados de una cantidad mucho mayor explican casi toda la varianza, es menos exigente computacionalmente transformar los datos y hacer la predicción en esas pocas dimensiones transformadas que utilizar el dataset original, además de obtener mejores modelos con una mínima pérdida de precisión.

Naturalmente, esto último sólo es aplicable cuando existen algunos features transformados que casi no explican la varianza. No es conveniente utilizar PCA en el caso que la pérdida de información no amerite el beneficio de reducir la dimensionalidad si la varianza se encuentra mucho más equitativamente distribuída. Tampoco es conveniente aplicar la reducción cuando el dataset es de cierto tamaño, debido al costo de las operaciones con matrices.

Si utilizamos PCA para un conjunto $X_{train}$ y hacemos un modelo para aproximar el mismo, ¿qué hacemos si tenemos nuevos datos de test $X_{test}$?

* Para los nuevos datos tenemos primero que **normalizarlos** restándole a cada feature la media del feature para el conjunto de entrenamiento y dividirlo por la desviación estándar de dicho conjunto (en el caso de utilizar z-score standarization).

* Tras eso, **deben ser transformados** con respecto a los componentes principales del conjunto de entrenamiento con **_pca.transform($X_{test}$)_**. A partir de ahí ya pueden ser utilizados con el modelo empleado para el conjunto $X$.

Por otra parte, nuestro dataset transformado puede ser devuelto a la matriz con los predictores originales con el método **pca.inverse_transform(X_transformed)**, quedando exactamente el mismo dataset que X si no se realizó ninguna modificación, y variando si alguna modificación fue realizada (por ejemplo, eliminar un feature de un componente principal que casi no aportaba a la explicación de la varianza se traduciría en una muy ligera variación en el dataset original).

## Clustering

* _Clustering_ se refiere a aquellos métodos que separan un conjunto de datos $X$ en subgrupos (_clústers_), de tal forma que los datos pertenecientes a cada grupo tengan entre sí la mayor relación posible.
* Dado que no consideramos las etiquetas $y$, estos grupos se infieren exclusivamente a partir de los features de nuestra matriz de datos $X$.
* Para entender clustering, es posible interpretarlo como un método que le asigna un vector similar a un label a cada observación de $X$, donde cada una de las ocurrencias de dicho vector representa el clúster de la correspondiente observación.

### K-Means Clustering

K-Means Clustering es un método iterativo y de propósito general muy simple que particiona las observaciones en un número predefinido $K$ de clústers.

* Cada uno de los datos será asignado a un y sólo un cluster.
* Funciona según el siguiente algoritmo:

Algoritmo: Dado un número K de clústers predefinido

1. Definir aleatoriamente cada uno de los $K$ centroides de cada uno de los $\{1,2,...,K\}$ clústers. El centroide es el vector promedio de cada uno de los vectores fila asignados al mismo clúster, excepto en el primer paso en donde se utilizan centroides aleatorios.
2. Repetir (hasta que no haya cambios en los clústers asignados)
    1. Asignar cada observación al clúster en donde algún centroide se encuentra más cerca (por distancia Euclideana).
    2. Calcular el centroide para cada uno de los $K$ clústers y reemplazar los nuevos centroides por los anteriores.
    
(Recordemos que un centroide o centro geométrico dado un conjunto de puntos es la posición "promedio" de los puntos, como ocurre con el punto central de la siguiente figura)

![](Figures/triangle_centroid.png)

Analíticamente:

Dados los vectores fila $X_1, X_2, \dots, X_n$, el vector centroide estará dado por

$$X_{centroide} = \frac{1}{n} (X_1 + X_2 + \dots + X_n)$$

Gráficamente...

![](Figures/k_means_example.png)

* Veamos cómo funciona el algoritmo K-Means en el iris dataset.
* Este es un ejemplo en el cuál no tiene mucho sentido hacer clústering debido a que tenemos los labels $y$, pero dado que es un conjunto conocido vamos a ver cómo convergen los distintos clústers.

In [ ]:
from sklearn.cluster import KMeans

for number_of_cluster in range(4):
    
    kmeans = KMeans(n_clusters=number_of_cluster+1, random_state=80)
    kmeans.fit(X)

    print("Número de clúster asignado a cada observación ", kmeans.labels_)
    print("Centroides de cada clúster ", kmeans.cluster_centers_)

    for i in range(np.max(kmeans.labels_)+1):
        plt.plot(X[kmeans.labels_==i,0], X[kmeans.labels_==i,1], 's', label="Cluster " + str(i+1))
        plt.plot(kmeans.cluster_centers_[i,0], kmeans.cluster_centers_[i,1], 'o', color='orange')

    plt.xlabel(iris.feature_names[0])
    plt.ylabel(iris.feature_names[1])
    plt.legend(loc='best', numpoints=1)
    plt.show()

* Ejemplo de problema con KMeans

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs

for number_of_cluster in range(3):

    # Creamos las tres "manchas" que forman nuestros datos para reflejar este ejemplo
    n_samples = 1500
    X, y = make_blobs(n_samples=n_samples, random_state=170)
    transformation = [[ 0.60834549, -0.63667341], [-0.40887718, 0.85253229]]
    X_aniso = np.dot(X, transformation)

    # Ajustamos nuestros datos a un modelo KMeans. Vemos lo que pasa cuando cambia el número de clústers
    km = KMeans(n_clusters=number_of_cluster+1, random_state=170)
    km.fit(X_aniso)
    km_pred = km.predict(X_aniso)

    plt.scatter(X_aniso[:, 0], X_aniso[:, 1], c=km_pred)
    plt.show()

#### Mixture Models

Los _Mixture Models_ son una variante de **Expectation Maximization** (EM), que son una familia de algoritmos empleados para encontrar parámetros desconocidos en modelos estadísticos. Muy simplificadamente EM en K-means se divide en los siguientes pasos:

* El **Expectation step** (E), donde se asigna cada observación al clúster más probable de acuerdo a una distribución dada con sus respectivos parámetros.
* El **Maximization step** (M), donde se optimizan los parámetros dadas las asignaciones realizadas en E, maximizando su **likelihood function** (muy simplificadamente, es una función que asigna una probabilidad a un evento dados sus parámetros estadísticos).
* Dada una semilla inicial, EM itera sucesivamente hasta que se alcanza estabilidad entre los pasos E-M (es decir que no cambian los parámetros / datos inferidos).

En los MM, se asume que los datos generados vienen de una **mezcla (_mixture_) de distribuciones probabilísticas de la misma familia (ej: Gaussianas) pero con distintos parámetros**. En nuestro ejemplo nos vamos a centrar en _Mixture of Gaussians_.

* Mixture of Gaussians puede ser pensado simplificadamente como una "generalización del KMeans" en donde se asume que los **centroides son formados a partir de distribuciones normales, además de incorporar información sobre la varianza de los datos**.
* Veamos cómo los mismos logran separar mucho mejor los datos de nuestro último ejemplo.

In [ ]:
#Modelo de Gaussian Mixture Model


### Conclusiones

* El aprendizaje no supervisado es una forma de aprendizaje donde el foco está centrado en extraer conocimiento sobre los datos con los que contamos, sin considerar la salida de los mismos.
* Hemos visto los dos enfoques principales: PCA y clustering.
* Con PCA, tomamos un dataset y, a partir de una transformación lineal, vemos cuán correlacionados están sus datos al contar con vectores ortogonales que los transforman.
* Con clustering, lo que hacemos es ver cómo los datos pueden dividirse en subgrupos, de modo tal que podamos entender mejor las propiedades subyacentes de los features.

Ambos enfoques nos sirven para obtener conocimiento desde una perspectiva distinta de la propuesta por las técnicas de aprendizaje supervisado.


* **Visión general sobre cuándo usar qué métodos de Aprendizaje No Supervisado y Supervisado:** http://scikit-learn.org/stable/tutorial/machine_learning_map/index.html